# 03 Error analysis (validation only)

Draft. The best run is chosen by **validation** Macro F1; test is not looked at here. Built by `scripts/error_analysis.py notebook`. Only English glosses (Gemini), labels and ids are shown; original comment text stays out of the notebook.

In [1]:
import json, pandas as pd
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
import sys; sys.path.insert(0, str(ROOT / "src"))
from terralens.evaluate import confusion_df, per_class_report
sample = pd.read_csv(ROOT / "data/interim/error_sample.csv", dtype=str, keep_default_na=False)
RUN = sample["run"].iloc[0]
res = json.loads((ROOT / "results" / f"{RUN}.json").read_text(encoding="utf-8"))
pred = pd.read_csv(ROOT / "results/predictions" / f"{RUN}_val.csv", dtype=str, keep_default_na=False)
print("Best run by val Macro F1:", RUN, "| val Macro F1", res["val"]["macro_f1"], "| val Against F1", res["val"]["against_f1"])

Best run by val Macro F1: e3_best_plus_bcd | val Macro F1 0.6741 | val Against F1 0.4706


## Confusion matrix and per-class scores (val)

In [2]:
print(confusion_df(pred['gold'], pred['pred']))
print(per_class_report(pred['gold'], pred['pred']).round(3))

pred     Favour  Against  None
gold                          
Favour       48        1    25
Against       4        8     7
None         27        6   396
         precision  recall     f1  support
Favour       0.608   0.649  0.627       74
Against      0.533   0.421  0.471       19
None         0.925   0.923  0.924      429


## Macro F1 per language, script and annotator tag (val)

In [3]:
for k in ('by_lang', 'by_script', 'by_tag'):
    print(k); display(pd.DataFrame(res['val'][k]).T)

by_lang


,n,macro_f1,against_f1
bn,165.0,0.5045,0.3333
hi,357.0,0.7011,0.5455


by_script


,n,macro_f1,against_f1
bengali,130.0,0.5004,0.3333
devanagari,150.0,0.7687,0.6667
latin,226.0,0.6130,0.4000
mixed,16.0,0.5877,0.0000


by_tag


,n,macro_f1,against_f1,accuracy
sarcasm,19.0,0.4154,0.4000,0.6842
implicit,44.0,0.2721,0.0000,0.5682
code_mixed,63.0,0.7943,0.6667,0.8730
romanised,269.0,0.6389,0.4000,0.8587
question,28.0,0.3273,0.0000,0.9643


## 40 misclassified validation comments

Sampled with seed 42, spread over gold -> prediction pairs. `category` is a hand-assigned error type (see `reports/error_categories.csv`).

In [4]:
gl = {}
p = ROOT / 'data/interim/error_glosses.jsonl'
for line in p.read_text(encoding='utf-8').splitlines():
    r = json.loads(line); gl[r['id']] = r['gloss']
cat = pd.read_csv(ROOT / 'reports/error_categories.csv', dtype=str) if (ROOT / 'reports/error_categories.csv').exists() else pd.DataFrame(columns=['id','category'])
view = sample.assign(gloss=sample['id'].map(gl)).merge(cat, on='id', how='left')
pd.set_option('display.max_colwidth', 160)
display(view[['label', 'pred', 'lang', 'script', 'tags', 'category', 'gloss']].rename(columns={'label': 'gold'}))

,gold,pred,lang,script,tags,category,gloss
0,Favour,None,hi,devanagari,"[""implicit""]",implicit stance,Human beings are releasing the most methane gas
1,Favour,None,hi,latin,"[""romanised""]",implicit stance,It is a reason for climate change .. excessive consumption and population
2,Favour,None,hi,latin,"[""implicit"", ""romanised""]",implicit stance,"If you mess with natural things, all this will happen"
3,None,Favour,hi,devanagari,[],solution or topic talk without stance,Pure vegetarianism is the only solution :check_mark_button:
4,Against,None,hi,latin,"[""romanised"", ""sarcasm""]",sarcasm,":face_with_tears_of_joy: Now these bastards are blaming animals, the scoundrels. Pollution is not caused by industrial activities :face_with_tears_of_joy: :..."
5,Favour,None,hi,latin,"[""implicit"", ""romanised""]",implicit stance,The beginning of destruction has started :crying_face: :crying_face: :crying_face: :crying_face: :folded_hands: :pensive_face: :folded_hands: :pensive_face:
6,None,Favour,bn,bengali,[],religious or moral framing,"Do not despair, people of the world, look at the mountain of your sins; with a little bit of mercy, your fate will change."
7,None,Favour,hi,latin,"[""code_mixed"", ""romanised""]","topic, not target (pollution, local environment)","In our village, a mini dairy is running near our house, breathing becomes difficult, but it is a source of income for them and Bihar's Sudha dairy purchases..."
8,None,Favour,hi,devanagari,"[""romanised""]","topic, not target (pollution, local environment)","This scientific fact is absolutely correct, the use of pesticides and fertilizers should be stopped completely"
9,Favour,Against,hi,devanagari,"[""code_mixed"", ""implicit""]","blame, politics or conspiracy",More global warming is being caused by the world guru and Adani. It is beyond understanding why poor animals are being dragged into this.


## Error categories

In [5]:
display(view.groupby('category').agg(n=('id', 'size'), pairs=('pair', lambda s: s.value_counts().to_dict())).sort_values('n', ascending=False))

,n,pairs
category,,
implicit stance,7,{'Favour->None': 7}
sarcasm,6,"{'Against->None': 4, 'Against->Favour': 1, 'Favour->None': 1}"
needs video context,5,{'Favour->None': 5}
weather facts without stance,4,"{'None->Favour': 3, 'None->Against': 1}"
"blame, politics or conspiracy",4,"{'None->Favour': 2, 'Favour->Against': 1, 'Against->Favour': 1}"
"topic, not target (pollution, local environment)",4,{'None->Favour': 4}
religious or moral framing,3,"{'None->Favour': 2, 'None->Against': 1}"
generic or off-topic,2,{'None->Favour': 2}
possible label noise,2,"{'None->Favour': 1, 'None->Against': 1}"
